In [6]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

DATA_PATH = r"E:\Bed-Time-Buddy--A-RAG-based-chatbot\Data"

print("Loading PDFs...")
loader = PyPDFDirectoryLoader(DATA_PATH)
data = loader.load()

print(f"Loaded {len(data)} pages")

print("Splitting into chunks...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

text_chunks = text_splitter.split_documents(data)

print(f"Created {len(text_chunks)} chunks")

Loading PDFs...
Loaded 366 pages
Splitting into chunks...
Created 1644 chunks


In [ ]:
from pinecone import Pinecone, ServerlessSpec
from langchain_community.vectorstores import Pinecone as PineconeVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings
import os

# 🔴 Your API key
PINECONE_API_KEY = "your-pinecone-api-key"
INDEX_NAME = "your-index-name"

# ✅ VERY IMPORTANT (LangChain needs this)
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

# Init client
pc = Pinecone(api_key=PINECONE_API_KEY)

# Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create index if not exists
if INDEX_NAME not in [i["name"] for i in pc.list_indexes()]:
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# Upload documents
docsearch = PineconeVectorStore.from_documents(
    text_chunks,
    embedding=embeddings,
    index_name=INDEX_NAME
)

print("✅ Upload success")

✅ Upload success


In [12]:
docs = retriever.get_relevant_documents("salt pepper sugar")
print("\n--- RETRIEVED DOCS ---")
print(docs)

e:\Bed-Time-Buddy--A-RAG-based-chatbot\venv\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(



--- RETRIEVED DOCS ---
[Document(page_content='seasoning and making things have a good taste.” \n“Ah, Sugar Bowl,” said the Salt Cellar, “I am glad to hear you \ntalk this way. For some time I have been afraid that you didn’t have \nenough character. I was very much afraid that you were becoming \ntoo sugary and too weak! \n“The Pepper Shaker would tell you, too, how much he thinks of you, \n65', metadata={'page': 96.0, 'source': 'E:\\Bed-Time-Buddy--A-RAG-based-chatbot\\Data\\365_bedtime_stories,_(IA_365bedtimestorie00bonn).pdf'}), Document(page_content='laugh and admire the nice, cheerful, friendly Polly Parrot. And her \nmistress was very proud of Polly! \nMARCH 21: Salt, Pepper and Sugar \nCREATURES and things aren’t to be admired who won’t take \nthe trouble to go out of their way to do nice things,” said \nSugar of the Sugar Bowl. “And as the Sugar Bowl can’t \ngo walking around looking for nice things to do at least it can ad¬ \nmire the Salt Cellar and the Pepper Shaker for th

In [18]:
from langchain_community.llms import Ollama
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_community.vectorstores import Pinecone as PineconeVectorStore

# ✅ Load existing vector DB
docsearch = PineconeVectorStore.from_existing_index(
    index_name="bed-time-buddy",
    embedding=embeddings
)

# ✅ Better retriever (less noise)
retriever = docsearch.as_retriever(search_kwargs={"k": 2})

# ✅ Lightweight LLM (works on low RAM)
llm = Ollama(
    model="phi3",
    temperature=0.3
)

# ✅ STRONG PROMPT (critical fix)
template = """

Answer ONLY using the given context.

Rules:
- Do NOT create a story
- Do NOT add new characters
- Do NOT assume anything
-Answer in simple language, like you're telling a story to a child
- If exact answer not found, say: "I don't know"

Context:
{context}

Question:
{question}

"""

PROMPT = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

# ✅ RAG chain
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=False
)

# ✅ Test query
query = "Tell me a story about A sun Parlor for birds"

result = qa.invoke({"query": query})

print("\n--- ANSWER ---")
print(result["result"])


--- ANSWER ---
Once upon a time in a world much like ours but filled with magic from the Sun itself lived Mr. and Mrs. Sun, who had created something very special called "A Sun Parlor." This wasn't just any room; it was made entirely of glass so that sunlight could shine through all day long for their little bird friends to enjoy.

Inside this glowing home were tiny doors too small even the tiniest birds couldn't squeeze into, but they had a secret passage called "the cellar." This was where Mr. and Mrs. Sun kept lots of yummy treats like apples that smelled sweet as honey, crumbs from bread to munch on during snack time, delicious water droplets for drinking, and many other goodies birds love so much!

But there was a big question in the air. How did all these little bird friends get down safely every day? They needed help with that because they couldn't fly as far or fast like their parents could. So Jack asked this very important question to Mr. and Mrs. Sun, hoping for an answer s